In [1]:
!pip install gtfparse
!pip install polars=='0.16.17'

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 27.5 MB/s eta 0:00:0000:0100:01
  Created wheel for gtfparse: filename=gtfparse-2.0.1-py3-none-any.whl size=15285 sha256=558744ba88872661628a89a7929712c4c0774fc6ebe7c6f776e190ac79b51ae2
  Stored in directory: /root/.cache/pip/wheels/91/da/d4/4168bc0aa594bfcda1ba95d81ea91552d506807d686ceb4e1e
Successfully built gtfparse
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 8.5 MB/s eta 0:00:00:00:0100:01
Reason for being yanked: <none given>
  Attempting uninstall: polars
    Found existing installation: polars 0.18.4
    Uninstalling polars-0.18.4:
      Successfully uninstalled polars-0.18.4


In [2]:
!pip install pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.1/39.1 MB 17.7 MB/s eta 0:00:0000:0100:01


In [3]:
!pip install anndata==0.8.0

In [1]:
from samalg import SAM
from Bio import SeqIO
from gtfparse import read_gtf
import pandas as pd
import pandas
import pyarrow
import pickle
import scanpy as sc
import numpy as np

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#open scdata to see how many gene matches you have
dat = sc.read_h5ad('../../Testing_Raw_Dat_RNASEQ_Joined/tot_dat_RI_genome.h5ad')

In [3]:
dat.var_names

Index(['A1CF', 'A4GALT', 'AAAS', 'AACS', 'AADAT', 'AAGAB', 'AAK1', 'AAMDC',
       'AAMP', 'AANAT',
       ...
       'ZSWIM7', 'ZSWIM8', 'ZSWIM9', 'ZUP1', 'ZW10', 'ZWILCH', 'ZWINT', 'ZXDC',
       'ZYX', 'ZZZ3'],
      dtype='object', length=23867)

In [3]:
#RI created using gffread -w -F
#AC and DR created using kb ref -i -g -f1
input_file = open("../../cDNA_fasta/DR_ncbi_cdna_03142025.fa")
gene_dict = {}
for item in SeqIO.parse(input_file, "fasta"):
    print(item)
    break

ID: NM_173235.3
Name: NM_173235.3
Description: NM_173235.3 gene_id:rpl24 gene_name: transcript_name: chr:NC_007112.7 start:6642 end:11878 strand:-
Number of features: 0
Seq('AGGGTTCATTCCTATGTCAAATATATGTTTACTTCAAAAAAATATTTTACTTTA...TCA', SingleLetterAlphabet())


In [10]:
#pull longest gene
input_file = open("../../cDNA_fasta/RI_gffread_F_raw_test_prateek.fa")
gene_dict = {}
for item in SeqIO.parse(input_file, "fasta"):
    #manually change gene_ID to match which field you would like to see in your BLAST table
    #for RI use 'gene=' and ';'
    #for AC and DR use 'gene_id:' and ' '
    gene_ID = item.description.split('gene=')[1].split(';')[0]
    if gene_ID in gene_dict.keys():
        if len(item.seq) > len(gene_dict[gene_ID].seq):
            gene_dict[gene_ID] = item
    else:
        gene_dict[gene_ID] = item
len(gene_dict)

32095

In [11]:
len(set(dat.var_names) & set(gene_dict.keys()))

23801

In [12]:
for item in gene_dict.keys():
    gene_dict[item].id = item
    gene_dict[item].name = item

with open("../../BLASTMAPPING/RI_ncbi_genome_curated_08292026_gffreadF.fa", "w") as handle:
    SeqIO.write(gene_dict.values(), handle, "fasta") 